In [ ]:
# %% [markdown]
# # 2. Model Training - Spam Classifier
# 
# **Steps:**
# 1. Data Cleaning
# 2. Text Preprocessing
# 3. Feature Extraction
# 4. Model Training
# 5. Evaluation

# %% [code]
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

# Download stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# %% [code]
# Load data
df = pd.read_csv('../data/spam.csv', encoding='latin-1')
df = df[['v1', 'v2']]
df.columns = ['label', 'message']

# Convert labels to binary
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

# %% [code]
# Text cleaning function
def clean_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Remove stopwords
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

# Apply cleaning
df['clean_message'] = df['message'].apply(clean_text)
print("Cleaned messages example:")
print(df[['message', 'clean_message']].head())

# %% [code]
# Vectorization
vectorizer = CountVectorizer(max_features=3000)
X = vectorizer.fit_transform(df['clean_message']).toarray()
y = df['label']

print(f"Feature matrix shape: {X.shape}")

# %% [code]
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

# %% [code]
# Train model
model = MultinomialNB()
model.fit(X_train, y_train)

# Save model and vectorizer
joblib.dump(model, '../models/spam_model.pkl')
joblib.dump(vectorizer, '../models/vectorizer.pkl')
print("Model saved successfully!")

# %% [code]
# Evaluate
y_pred = model.predict(X_test)

print("Model Performance:")
print("="*50)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# %% [code]
# Confusion Matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Ham', 'Spam'], 
            yticklabels=['Ham', 'Spam'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

# %% [code]
# Test with custom messages
test_messages = [
    "Free entry to win £1000 cash! Call now!",
    "Hey, are we meeting tomorrow for lunch?",
    "Congratulations! You've won a free iPhone. Click here to claim.",
    "Mom, can you pick me up at 5 pm?"
]

def predict_message(msg):
    clean_msg = clean_text(msg)
    vector = vectorizer.transform([clean_msg]).toarray()
    pred = model.predict(vector)[0]
    return "SPAM" if pred == 1 else "HAM"

print("\nCustom Predictions:")
print("="*50)
for msg in test_messages:
    print(f"{msg[:50]}... → {predict_message(msg)}")